# 확장 시각화 — 텍스트 워드클라우드(WordCloud) · 지도 시각화(Folium) 정답지

이 노트북은 `Visualization_5_wordcloud_folium_Practice.ipynb`의 모든 연습문제(1 ~ 12번)에 대한 **모범 시각화 코드와 비정형 데이터 분석 해석**을 수록한 정답지입니다.
텍스트 토큰화(`split`, `strip`), 불용어(Stopwords) 정제, 단어 빈도 카운팅(`Counter`), 워드클라우드 생성 및 커스터마이징(`WordCloud`), Folium 대화형 지도(`folium.Map`), 배경 타일(`tiles`), 공간 필터링(`between`), 마커 클러스터(`MarkerCluster`) 및 HTML 저장 검증을 체계적으로 다룹니다.


## 처음 배우는 분을 위한 안내

토큰은 문장을 쪼갠 단어 단위, 불용어는 분석에서 빼기로 정한 단어, Counter는 출현 횟수를 세는 도구입니다. 여기서는 공백 분리만 사용하며 형태소 분석은 필요하지 않습니다. 위도는 남북 위치, 경도는 동서 위치이며 지도 좌표는 [위도, 경도] 순서입니다.

**학습 순서:** 문제 1 ~ 4는 필수 기초, 5 ~ 12는 선택 도전입니다. 5 ~ 6번은 한글 글꼴이, 7번 이후 지도 화면은 인터넷이 필요할 수 있습니다. 인터넷이 없어도 마커 데이터와 HTML 생성 여부는 확인할 수 있습니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

**지도 화면 주의:** 배경 지도와 Leaflet 스크립트는 외부 서버를 사용합니다. 네트워크가 막혀 빈 배경이 나와도 마커 개수·좌표와 HTML 저장 성공을 따로 확인하세요. 글꼴 파일이 없으면 5 ~ 6번 워드클라우드는 잠시 건너뛰고 1 ~ 4번 표와 7번 이후 지도를 진행할 수 있습니다.

## 실행 준비

Python 3와 `pandas`, `numpy`, `matplotlib`, `seaborn`이 필요합니다(확장 시각화 노트북은 `wordcloud`, `folium` 추가 필요). 없는 경우 별도 코드 셀에서 `%pip install pandas numpy matplotlib seaborn wordcloud folium`을 한 번 실행하세요. 이 노트북은 외부 CSV 없이 실습 데이터가 자동 생성됩니다.

Colab 또는 Jupyter에서 `.ipynb`를 열고 **준비 셀부터 순서대로** 실행하세요. 아래 코드는 한글 폰트 설정과 실습 데이터 생성만 수행합니다.

In [ ]:
import platform
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import folium
from folium.plugins import MarkerCluster

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
plt.rcParams['axes.unicode_minus'] = False

# 지정 글꼴이 없는 환경에서는 설치된 한글 글꼴 파일을 FONT_PATH에 지정합니다.
from pathlib import Path
if not Path(FONT_PATH).is_file():
    candidates = [p for pattern in ('/System/Library/Fonts/*Apple*','/usr/share/fonts/truetype/nanum/*.ttf') for p in __import__('glob').glob(pattern) if Path(p).is_file()]
    if candidates:
        FONT_PATH = candidates[0]
    else:
        raise FileNotFoundError('한글 글꼴 파일이 필요합니다. 글꼴 파일을 올린 뒤 FONT_PATH를 해당 경로로 설정하세요.')

# 1. 워드클라우드 실습용 고객 리뷰·문장 리스트
review_lines = [
    "배송 빠름 포장 깔끔 가격 만족 합니다.",
    "배송 빠름 품질 최고 재구매 의사 있습니다,",
    "가격 합리적 배송 빠름 상담 친절 합니다.",
    "포장 깔끔 품질 만족 배송 최고 입니다.",
    "품질 최고 가격 만족 포장 안전 합니다."
]

# 2. Folium 지도 실습용 매장 위치 데이터 (국가 및 위경도 포함, 8행)
stores_df = pd.DataFrame({
    'Store.Name': [
        '시청역점', '광화문점', '을지로입구점', '명동역점',
        '정동점', '강남역점', '부산서면점', '도쿄신주쿠점'
    ],
    'Country': ['KR', 'KR', 'KR', 'KR', 'KR', 'KR', 'KR', 'JP'],
    'Latitude': [37.5658, 37.5715, 37.5660, 37.5609, 37.5672, 37.4979, 35.1578, 35.6938],
    'Longitude': [126.9772, 126.9765, 126.9827, 126.9858, 126.9721, 127.0276, 129.0592, 139.7034]
})

# 3. 공공시설(의류수거함) 위치 현황 가상 데이터 (결측 1행 포함, 6행)
collection_bins = pd.DataFrame({
    'bin_id': ['B01', 'B02', 'B03', 'B04', 'B05', 'B06'],
    'dong': ['둔산동', '둔산동', '갈마동', '월평동', '갈마동', '둔산동'],
    'latitude': [36.3504, 36.3521, 36.3482, 36.3585, None, 36.3512],
    'longitude': [127.3845, 127.3880, 127.3720, 127.3615, None, 127.3860]
})

## 문제 1. 문장 분리(`split`)와 문장부호 제거(`strip`) [필수 기초]

`review_lines`의 각 문장을 공백 기준으로 나누고(`split()`), 단어 양끝의 마침표(`.`)와 쉼표(`,`)를 제거(`strip('.,')`)하여 순서대로 모은 리스트 `raw_tokens`를 만드세요. 전체 단어 개수와 앞 10개 단어를 출력하세요.

**점검 기준:** 5개 문장(각 7어절)에서 문장부호가 제거된 총 35개의 단어가 `raw_tokens`에 저장됩니다.

In [ ]:
# 문제 1 풀이
raw_tokens = []
for line in review_lines:
    for word in line.split():
        clean_word = word.strip('.,')
        if clean_word:
            raw_tokens.append(clean_word)

print(f"전체 추출 단어 개수: {len(raw_tokens)}개")
print("앞 10개 단어        :", raw_tokens[:10])


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 5개 리뷰 문장(각 7어절)에 대해 공백 분리와 문장부호(`.`, `,`) 제거를 적용한 결과 총 **35개**의 원시 토큰이 누락 없이 순서대로 추출되었습니다.
2. '합니다.', '있습니다,'처럼 단어 끝에 붙은 구두점을 사전에 제거하지 않으면 동일한 의미의 단어가 다른 토큰으로 분리되어 빈도 집계가 왜곡되므로, 정규화(Normalization) 전처리가 필수적입니다.


## 문제 2. 분석에서 제외할 단어 제거 후 핵심 단어 리스트 만들기 [필수 기초]

`raw_tokens`에서 이 예제에서 제외하기로 정한 `'합니다'`, `'있습니다'`, `'입니다'` 세 단어를 제외한 리스트 `clean_tokens`를 만드세요. 제거 전후 단어 수를 비교하세요.

**점검 기준:** 불용어 5개가 제외되어 `clean_tokens`의 총 단어 수는 30개, 고유 단어 수는 14개입니다.

In [ ]:
# 문제 2 풀이
stopwords = {'합니다', '있습니다', '입니다'}
clean_tokens = [w for w in raw_tokens if w not in stopwords]

print(f"제거 전 단어 수: {len(raw_tokens)}개")
print(f"제거 후 단어 수: {len(clean_tokens)}개 (제거된 불용어: {len(raw_tokens) - len(clean_tokens)}개)")
print(f"고유 단어 수    : {len(set(clean_tokens))}개")
print("정제된 단어 목록:", clean_tokens[:12], "...")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 서술어 어미인 불용어 5개('합니다' 3개, '있습니다' 1개, '입니다' 1개)를 성공적으로 배제하여 핵심 의미를 지닌 단어 **30개(고유 단어 14개)**를 확보했습니다.
2. 문법적 기능어(불용어)를 걸러냄으로써 리뷰의 실질적 감성이나 속성 키워드('배송', '빠름', '품질', '가격', '만족' 등)의 출현 빈도가 시각화에서 정확히 강조되도록 준비되었습니다.


## 문제 3. `Counter`로 단어 빈도 계산 및 내림차순 정렬 [필수 기초]

`collections.Counter`를 사용해 `clean_tokens`의 단어별 출현 빈도를 세고, 빈도가 높은 순서(`reverse=True`)로 정렬한 리스트 `wc_sorted`를 출력하세요.

**점검 기준:** 가장 많이 등장한 단어는 `'배송'(4회)`이며, 이어서 `'빠름'`, `'포장'`, `'가격'`, `'만족'`, `'품질'`, `'최고'`가 각 3회씩, `'깔끔'`이 2회, `'재구매'`, `'의사'`, `'합리적'`, `'상담'`, `'친절'`, `'안전'`이 각 1회씩 등장합니다.

In [ ]:
# 문제 3 풀이
token_counts = Counter(clean_tokens)
wc_sorted = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)

print("=== 단어별 출현 빈도 (내림차순) ===")
for word, freq in wc_sorted:
    print(f"'{word}': {freq}회", end=", ")
print()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **단어 빈도 집계 결과:**
   - 최빈 단어: **`'배송'(4회)`**
   - 3회 출현 단어: `'빠름'`, `'포장'`, `'가격'`, `'만족'`, `'품질'`, `'최고'` (총 6개)
   - 2회 출현 단어: `'깔끔'` (1개)
   - 1회 출현 단어: `'재구매'`, `'의사'`, `'합리적'`, `'상담'`, `'친절'`, `'안전'` (총 6개)
2. 고객들이 작성한 리뷰 텍스트의 중심 화두가 '배송의 신속성'과 '포장 및 상품 품질의 만족도'에 강하게 집중되어 있음을 정량적으로 파악했습니다.


## 문제 4. 단어 빈도 데이터프레임 변환 및 `{단어: 빈도}` 딕셔너리 생성 [필수 기초]

`wc_sorted`를 컬럼명이 `['word', 'freq']`인 데이터프레임 `wcdf`로 만들고, `'word'`를 인덱스로 설정한 뒤 `.to_dict()`를 사용하여 `{단어: 빈도}` 형태의 딕셔너리 `word_freq`를 생성하세요.

**점검 기준:** `wcdf`는 `14행 × 2열`이며, `word_freq['배송'] == 4`, `word_freq['빠름'] == 3`, `word_freq['깔끔'] == 2`인 딕셔너리가 생성됩니다.

In [ ]:
# 문제 4 풀이
wcdf = pd.DataFrame(wc_sorted, columns=['word', 'freq'])
word_freq = wcdf.set_index('word')['freq'].to_dict()

print(f"wcdf shape: {wcdf.shape} (14행 x 2열)")
display(wcdf.head(6))

print("\nword_freq 딕셔너리 점검:")
print(f"word_freq['배송'] == {word_freq['배송']}")
print(f"word_freq['빠름'] == {word_freq['빠름']}")
print(f"word_freq['깔끔'] == {word_freq['깔끔']}")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 정렬된 단어 빈도 리스트를 14행 × 2열의 데이터프레임 `wcdf`로 구조화한 후, 워드클라우드 라이브러리의 표준 입력 형식인 `{단어: 빈도}` 딕셔너리 `word_freq`로 변환했습니다.
2. `word_freq['배송'] == 4`, `word_freq['빠름'] == 3`, `word_freq['깔끔'] == 2` 검증 결과 점검 기준과 정확히 일치함을 확인했습니다.


## 문제 5. 기본 워드클라우드 생성 및 화면 표시 [선택 도전]

`WordCloud(font_path=FONT_PATH, width=800, height=500).generate_from_frequencies(word_freq)`로 워드클라우드 객체를 생성하고 `plt.imshow()`와 `plt.axis('off')`로 출력하세요.

**점검 기준:** 빈도가 가장 높은 `'배송'`이 가장 크게 배치되고 2회 출현 단어들이 중간 크기로 배치된 워드클라우드 이미지가 생성됩니다.

In [ ]:
# 문제 5 풀이
wc = WordCloud(
    font_path=FONT_PATH,
    width=800,
    height=500,
    random_state=42
).generate_from_frequencies(word_freq)

plt.figure(figsize=(9, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.tight_layout()
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. `generate_from_frequencies()` 메서드를 통해 빈도수에 비례하여 글자 크기가 결정되는 기본 워드클라우드를 생성했습니다.
2. **시각적 계층 구조:** 최빈 단어인 **`배송`**이 중앙에 가장 큰 폰트로 배치되고, 그 주변에 **`빠름`, `품질`, `최고`, `만족`, `가격`, `포장`**이 2순위 크기로 균형 있게 배치되어 텍스트 데이터의 핵심 요약이 한눈에 전달됩니다.


## 문제 6. 워드클라우드 배경색(`background_color`)과 컬러맵(`colormap`) 커스터마이징 [선택 도전]

`background_color='white'`, `colormap='autumn'`, `width=900`, `height=500` 옵션을 적용해 새로운 워드클라우드를 생성하고 제목 `'Word Frequency'`를 붙여 출력하세요.

**점검 기준:** 흰색 배경 위에 붉은/주황 계열(`autumn`) 색상으로 키워드가 선명하게 시각화됩니다.

In [ ]:
# 문제 6 풀이
wc_custom = WordCloud(
    font_path=FONT_PATH,
    background_color='white',
    colormap='autumn',
    width=900,
    height=500,
    random_state=42
).generate_from_frequencies(word_freq)

plt.figure(figsize=(9, 5))
plt.imshow(wc_custom, interpolation='bilinear')
plt.axis('off')
plt.title('Word Frequency', fontsize=15, pad=12)
plt.tight_layout()
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 기본 검은색 배경 대신 깔끔한 `background_color='white'`를 적용하고, 붉은색과 주황색 계열의 `colormap='autumn'`을 적용하여 비즈니스 보고서에 적합한 가독성 높은 워드클라우드를 완성했습니다.
2. 흰색 배경은 인쇄나 슬라이드 삽입 시 시각적 피로도를 낮추고 텍스트의 윤곽을 선명하게 부각해 줍니다.


## 문제 7. 서울시청 중심 Folium 기본 지도와 빨간색 마커 생성 [선택 도전]

서울시청 좌표 `city_hall = [37.5662952, 126.9779451]`을 중심으로 `zoom_start=14`인 `folium.Map` 객체 `m1`을 생성하고, `tooltip='서울시청'`, `popup='서울시청 위치'`, `icon=folium.Icon(color='red')`인 마커를 추가하세요.

**점검 기준:** `m1` 지도 객체가 생성되며 중심 좌표 `[37.5662952, 126.9779451]`에 빨간색 핀 마커 1개가 등록됩니다.

In [ ]:
# 문제 7 풀이
city_hall = [37.5662952, 126.9779451]

m1 = folium.Map(location=city_hall, zoom_start=14)

folium.Marker(
    location=city_hall,
    tooltip='서울시청',
    popup='서울시청 위치',
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(m1)

print("m1 지도 객체 생성 완료:", type(m1))


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 서울시청 위경도 좌표 `[37.5662952, 126.9779451]`을 중심점으로 하는 확대 수준 `zoom_start=14`의 Folium 대화형 지도를 생성했습니다.
2. 마우스 호버 시 `'서울시청'` 말풍선(`tooltip`)이 나타나고, 클릭 시 `'서울시청 위치'` 팝업(`popup`)이 열리는 빨간색 핀 마커를 성공적으로 렌더링했습니다.


## 문제 8. 배경 지도 타일(`tiles='Esri.WorldImagery'`) 변경 비교 [선택 도전]

서울시청 좌표에 대해 `tiles='Esri.WorldImagery'`(항공·위성 사진 타일) 옵션을 지정한 지도 `m_sat`을 생성하고 서울시청 마커를 추가한 뒤 일반 지도 타일과의 차이를 적으세요.

**점검 기준:** `Esri.WorldImagery` 타일이 적용되어 도로망 지도 대신 위성/항공 영상이 배경으로 표시됩니다(단, 외부 타일 서비스는 제공 정책에 따라 영구 사용이 보장되지 않을 수 있음).

In [ ]:
# 문제 8 풀이
m_sat = folium.Map(location=city_hall, zoom_start=14, tiles='Esri.WorldImagery')

folium.Marker(
    location=city_hall,
    tooltip='서울시청 (위성 지도)',
    popup='서울시청 위치',
    icon=folium.Icon(color='red')
).add_to(m_sat)

print("m_sat 위성 지도 객체 생성 완료 (tiles='Esri.WorldImagery')")


**결과 해석 / 근거 / 새로 알게 된 점:**

### 기본 타일(OpenStreetMap) vs 위성 타일(Esri.WorldImagery) 비교
1. **표현 내용의 차이:**
   - 기본 타일은 도로망, 건물 외곽선, 지하철역 등 추상화된 벡터 지리 정보를 도로 중심으로 깔끔하게 보여줍니다.
   - 반면 `Esri.WorldImagery` 타일은 실제 지형의 산림, 하천, 실제 건물 옥상과 도로 아스팔트가 찍힌 고해상도 항공/위성 사진을 배경으로 보여줍니다.
2. **실무적 유의점:** 외부 상업 위성 타일 서비스는 라이선스 정책 변경이나 네트워크 차단에 따라 장기적 안정성이 변동될 수 있으므로, 목적(상권 분석 vs 도로 안내)에 맞춰 신중히 선택해야 합니다.


## 문제 9. 위도·경도 사각형 범위(`between`)로 서울 시청 주변 매장 필터링 [선택 도전]

`stores_df`에서 국가가 한국(`Country == 'KR'`)이고 위도가 `37.55 ~ 37.58`(`between(37.55, 37.58)`), 경도가 `126.96 ~ 126.99`(`between(126.96, 126.99)`)인 행만 선택해 `['Store.Name', 'Latitude', 'Longitude']` 열을 가진 `seoul_stores` 복사본을 만드세요.

**점검 기준:** 전체 8개 매장 중 도쿄(`JP`), 부산(`35.15`), 강남역(`37.49`) 매장이 제외되어 시청역점·광화문점·을지로입구점·명동역점·정동점 총 5개 매장이 선택됩니다.

In [ ]:
# 문제 9 풀이
cond_seoul = (
    (stores_df['Country'] == 'KR') &
    (stores_df['Latitude'].between(37.55, 37.58)) &
    (stores_df['Longitude'].between(126.96, 126.99))
)

seoul_stores = stores_df.loc[cond_seoul, ['Store.Name', 'Latitude', 'Longitude']].copy()

print(f"필터링된 매장 수: {len(seoul_stores)}개")
display(seoul_stores)


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **지리적 Bounding Box 필터링 결과:**
   - 원본 8개 매장 중 일본 도쿄 매장(`Country == 'JP'`), 부산 서면점(위도 35.15), 서울 강남역점(위도 37.49) 3개 매장이 정확히 배제되었습니다.
   - 서울 시청 주변 중심 업무 지구에 위치한 **시청역점, 광화문점, 을지로입구점, 명동역점, 정동점 총 5개 매장**이 선택되었습니다.
2. `series.between(하한, 상한)` 메서드를 활용하여 위도와 경도의 사각형 지리 영역을 간결하고 명확하게 필터링할 수 있습니다.


## 문제 10. `MarkerCluster`와 `iterrows()`로 매장 마커 일괄 추가하기 [선택 도전]

서울시청 중심 지도 `m`에 `MarkerCluster(name='Starbucks')`를 추가하고, `seoul_stores.iterrows()` 반복문을 사용해 5개 매장의 위치 마커(`tooltip=row['Store.Name']`, `popup=folium.Popup(row['Store.Name'], max_width=250)`)를 클러스터에 추가하세요.

**점검 기준:** 클러스터 객체 안에 5개의 매장 마커가 등록되며, 지도 축소 시 군집 배지로 묶이고 확대 시 개별 매장 마커로 펼쳐집니다.

**차근차근:** 새 folium.Map을 m에 저장합니다. cluster=MarkerCluster(...).add_to(m)으로 묶음을 만들고, for _, row in seoul_stores.iterrows()로 한 행씩 읽습니다. 각 행의 [Latitude, Longitude]로 Marker를 만들어 cluster에 add_to 합니다.

In [ ]:
# 문제 10 풀이
m = folium.Map(location=city_hall, zoom_start=14)
cluster = MarkerCluster(name='Starbucks').add_to(m)

for idx, row in seoul_stores.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        tooltip=row['Store.Name'],
        popup=folium.Popup(row['Store.Name'], max_width=250),
        icon=folium.Icon(color='green', icon='coffee', prefix='fa')
    ).add_to(cluster)

print("MarkerCluster에 5개 매장 등록 완료!")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. `seoul_stores.iterrows()` 루프를 통해 5개 매장의 개별 좌표에 초록색 커피 아이콘 마커를 일괄 생성하여 `MarkerCluster`에 바인딩했습니다.
2. **마커 클러스터의 효용:** 지도를 축소하면 밀집된 마커들이 숫자 '5'가 적힌 원형 클러스터 배지로 합쳐져 화면 난잡도를 방지하고, 지도를 확대하면 개별 매장 위치로 자연스럽게 분리되어 수천 수만 개의 거대 지리 데이터도 쾌적하게 탐색할 수 있습니다.


## 문제 11. 공공시설(의류수거함) 결측 좌표 제거 및 동별 색상 구분 마커 표시 [선택 도전]

`collection_bins`에서 위도·경도 결측치를 제거한 `valid_bins`를 만들고, 중심 좌표 `[36.351, 127.380]`(`zoom_start=14`) 지도에 `'둔산동'`은 파란색(`'blue'`), 그 외 동은 초록색(`'green'`) 아이콘 마커로 표시하세요.

**점검 기준:** 결측 1행(`B05`)이 제외되어 총 5개 수거함이 지도에 표시되며, 둔산동 3개(`B01, B02, B06`)는 파란색, 갈마동·월평동 2개(`B03, B04`)는 초록색 마커로 구분됩니다.

In [ ]:
# 문제 11 풀이
valid_bins = collection_bins.dropna(subset=['latitude', 'longitude']).copy()
print(f"결측치 제거 전: {len(collection_bins)}개, 제거 후: {len(valid_bins)}개 (제거된 행: B05)")

m_bins = folium.Map(location=[36.351, 127.380], zoom_start=14)

for idx, row in valid_bins.iterrows():
    marker_color = 'blue' if row['dong'] == '둔산동' else 'green'
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        tooltip=f"{row['bin_id']} ({row['dong']})",
        popup=f"수거함 ID: {row['bin_id']}<br>동: {row['dong']}",
        icon=folium.Icon(color=marker_color, icon='trash', prefix='fa')
    ).add_to(m_bins)

print("공공시설(의류수거함) 5개 마커 렌더링 완료!")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 좌표가 누락된 B05번 수거함 1건을 `dropna`로 사전 제거하여 런타임 매핑 오류를 방지했습니다.
2. 행정동 속성에 따라 조건부 색상 분기를 적용하여, **둔산동 3개(`B01, B02, B06`)는 파란색**, **갈마동·월평동 2개(`B03, B04`)는 초록색** 마커로 표시함으로써 관할 동별 인프라 분포를 시각적으로 즉각 구분할 수 있습니다.


## 문제 12. 완성된 Folium 지도를 HTML 파일로 저장하고 검증하기 [선택 도전]

문제 10에서 만든 지도 `m`을 `m.save('seoul_stores_map.html')`로 저장하고, `pathlib.Path('seoul_stores_map.html').exists()`와 파일 크기(`stat().st_size`)를 출력해 정상 저장 여부를 확인하세요.

**점검 기준:** `seoul_stores_map.html` 파일이 생성되며 파일 크기가 0보다 큰 HTML 문서(Leaflet 스크립트 및 마커 좌표 포함)로 저장됩니다.

In [ ]:
# 문제 12 풀이
from pathlib import Path

html_path = Path('seoul_stores_map.html')
m.save(str(html_path))

is_file_saved = html_path.exists()
file_size_bytes = html_path.stat().st_size if is_file_saved else 0

print(f"HTML 저장 성공 여부 : {is_file_saved}")
print(f"저장된 파일 절대 경로: {html_path.resolve()}")
print(f"HTML 파일 크기       : {file_size_bytes:,} bytes")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **HTML 파일 생성 검증:** 문제 10에서 완성한 Folium 지도 객체 `m`을 `seoul_stores_map.html`로 저장한 결과, 파일 존재 여부(`exists()`)가 `True`이며 파일 크기가 **수십 KB(약 10,000 ~ 20,000 bytes 이상)**로 정상 저장되었음을 확인했습니다.
2. **독립 실행 웹 페이지:** Folium이 생성한 HTML 문서는 Leaflet.js 라이브러리와 스타일시트, 5개 매장의 지리 좌표 데이터가 자체 내장되어 있어, 파이썬 설치가 없는 환경에서도 크롬이나 사파리 등 일반 웹 브라우저로 열면 완전히 독립적으로 동작하는 인터랙티브 지도를 고객이나 동료에게 즉시 공유할 수 있습니다.


## 선택 도전: 단어 네 개를 표로 검산하기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
extra_words = ['배송','배송','가격','포장']
```

### 진행 순서

1. Counter로 단어 출현 횟수를 셉니다.
2. 빈도 합계와 고유 단어 수를 출력합니다.
3. 가장 자주 등장하는 단어를 적습니다. 이 선택 도전은 폰트·지도·외부 파일 없이 표만으로 완료합니다.

**완료 기준:** 배송 2회, 가격·포장 각각 1회. 빈도 합계 4, 고유 단어 3개.

### 선택 도전 풀이

```python
from collections import Counter
extra_counts=Counter(extra_words);print(extra_counts)
print(sum(extra_counts.values()),len(extra_counts))
assert extra_counts['배송']==2 and sum(extra_counts.values())==4 and len(extra_counts)==3
```

## 제출 전 확인

- 준비 셀과 풀이를 위에서 아래로 실행했나요?
- 그래프에 제목(`title`), 축 이름(`xlabel`, `ylabel`), 다중 계열인 경우 범례(`legend`)를 표시했나요?
- 모든 점검 기준에 대해 수치 근거와 시각적 특징을 함께 남겼나요?
- 결측값 처리 여부와 표본 크기(또는 구간/윈도우 설정)를 확인했나요?
- 원본 데이터프레임 변수를 덮어쓰지 않고 안전하게 실습했나요?